In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import sqlite3
from scipy import stats
from itertools import combinations
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor, XGBClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.datasets import load_iris

Case Study:

John is the Head of Product at an Alberta, Canada telecommunications company.

**He wants to know the following information:**

Which customers are most likely to leave the company and why

How many people are we losing each year (on average)

What are your recommendations (if any)

In [2]:
alca_df = pd.read_csv("raw_cust_data.csv")

## Exploring Data

In [3]:
alca_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0.0,Yes,No,1.0,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,NaN,0.0,No,No,34.0,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0.0,No,No,2.0,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0.0,No,No,45.0,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,NaN,42.30,1840.75,No
4,NaN,0.0,No,No,2.0,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
alca_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            3463 non-null   object 
 1   SeniorCitizen     7040 non-null   float64
 2   Partner           6692 non-null   object 
 3   Dependents        6684 non-null   object 
 4   tenure            6359 non-null   float64
 5   PhoneService      7041 non-null   object 
 6   MultipleLines     7039 non-null   object 
 7   InternetService   7037 non-null   object 
 8   OnlineSecurity    7041 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7040 non-null   object 
 11  TechSupport       7042 non-null   object 
 12  StreamingTV       7042 non-null   object 
 13  StreamingMovies   7040 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7041 non-null   object 
 16  PaymentMethod     3565 non-null   object 


In [6]:
alca_df.isnull().sum()

gender              3580
SeniorCitizen          3
Partner              351
Dependents           359
tenure               684
PhoneService           2
MultipleLines          4
InternetService        6
OnlineSecurity         2
OnlineBackup           0
DeviceProtection       3
TechSupport            1
StreamingTV            1
StreamingMovies        3
Contract               0
PaperlessBilling       2
PaymentMethod       3478
MonthlyCharges         0
TotalCharges           2
Churn                  2
dtype: int64

In [7]:
alca_df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7040.000000,6359.000000,7043.000000
mean,0.162216,32.366567,64.761692
std,0.368675,24.586235,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [8]:
alca_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0.0,Yes,No,1.0,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,NaN,0.0,No,No,34.0,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0.0,No,No,2.0,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0.0,No,No,45.0,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,NaN,42.30,1840.75,No
4,NaN,0.0,No,No,2.0,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [9]:
alca_df['TotalCharges'] = pd.to_numeric(alca_df['TotalCharges'], errors='coerce')

In [10]:
alca_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            3463 non-null   object 
 1   SeniorCitizen     7040 non-null   float64
 2   Partner           6692 non-null   object 
 3   Dependents        6684 non-null   object 
 4   tenure            6359 non-null   float64
 5   PhoneService      7041 non-null   object 
 6   MultipleLines     7039 non-null   object 
 7   InternetService   7037 non-null   object 
 8   OnlineSecurity    7041 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7040 non-null   object 
 11  TechSupport       7042 non-null   object 
 12  StreamingTV       7042 non-null   object 
 13  StreamingMovies   7040 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7041 non-null   object 
 16  PaymentMethod     3565 non-null   object 


In [11]:
alca_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0.0,Yes,No,1.0,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,NaN,0.0,No,No,34.0,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0.0,No,No,2.0,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0.0,No,No,45.0,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,NaN,42.30,1840.75,No
4,NaN,0.0,No,No,2.0,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [12]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

cols_to_encode = ['Partner', 'Dependents', 'PhoneService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn']
encoders = {}   # store to inverse_transform later

for col in cols_to_encode:
    le = LabelEncoder()
    alca_df[col] = le.fit_transform(alca_df[col])
    encoders[col] = le


In [13]:
alca_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0.0,1,0,1.0,0,No phone service,DSL,0,2,0,0,0,0,Month-to-month,1,Electronic check,29.85,29.85,0
1,NaN,0.0,0,0,34.0,1,No,DSL,2,0,2,0,0,0,One year,0,Mailed check,56.95,1889.50,0
2,Male,0.0,0,0,2.0,1,No,DSL,2,2,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15,1
3,Male,0.0,0,0,45.0,0,No phone service,DSL,2,0,2,2,0,0,One year,0,NaN,42.30,1840.75,0
4,NaN,0.0,0,0,2.0,1,No,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65,1


## Splitting Data

In [ ]:
X = alca_df[[]]

y = alca_df[]